1.Ingestion Pipeline  [Knowledge Construction]

In [1]:
import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv()

c:\Users\Mallem Kondaiah\OneDrive\Documents\RAG\RAG_IMPLEMENTATION\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
#step 1 : Load all text files from the docs directory
loader = DirectoryLoader(
        path='docs',
        glob="*.txt",
        loader_cls=TextLoader,
        loader_kwargs={'encoding': 'utf-8'}
    )

documents = loader.load()
for i, doc in enumerate(documents[:2]):  # Show first 2 documents
        print(f"\nDocument {i+1}:")
        print(f"  Source: {doc.metadata['source']}")
        print(f"  Content length: {len(doc.page_content)} characters")
        print(f"  Content preview: {doc.page_content[:100]}...")
        print(f"  metadata: {doc.metadata}")


Document 1:
  Source: docs\Google.txt
  Content length: 232201 characters
  Content preview: ﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and t...
  metadata: {'source': 'docs\\Google.txt'}

Document 2:
  Source: docs\Microsoft.txt
  Content length: 201014 characters
  Content preview: ﻿Microsoft
Microsoft Corporation is an American multinational Microsoft Corporation
corporation and ...
  metadata: {'source': 'docs\\Microsoft.txt'}


In [3]:
# step 2 : Split documents into smaller chunks with overlap
chunk_size=1000
chunk_overlap=0
text_splitter = CharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap
    )
    
chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks[:5]):
            print(f"\n--- Chunk {i+1} ---")
            print(f"Source: {chunk.metadata['source']}")
            print(f"Length: {len(chunk.page_content)} characters")
            print(f"Content:")
            print(chunk.page_content)
            print("-" * 50)

Created a chunk of size 1055, which is longer than the specified 1000
Created a chunk of size 1436, which is longer than the specified 1000
Created a chunk of size 1039, which is longer than the specified 1000
Created a chunk of size 1078, which is longer than the specified 1000
Created a chunk of size 1043, which is longer than the specified 1000
Created a chunk of size 1019, which is longer than the specified 1000
Created a chunk of size 1068, which is longer than the specified 1000
Created a chunk of size 1211, which is longer than the specified 1000
Created a chunk of size 1450, which is longer than the specified 1000
Created a chunk of size 1762, which is longer than the specified 1000
Created a chunk of size 1038, which is longer than the specified 1000
Created a chunk of size 1120, which is longer than the specified 1000
Created a chunk of size 1076, which is longer than the specified 1000
Created a chunk of size 1090, which is longer than the specified 1000
Created a chunk of s


--- Chunk 1 ---
Source: docs\Google.txt
Length: 600 characters
Content:
﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and technology
company focusing on online advertising, search engine
technology, cloud computing, computer software,
quantum computing, e-commerce, consumer
electronics, and artificial intelligence (AI).[9] It has
been referred to as "the most powerful company in the The Google logo used since 2015
world" by the BBC[10] and is one of the world's most
valuable brands.[11][12][13] Google's parent company,
Alphabet Inc., is one of the five Big Tech companies
alongside Amazon, Apple, Meta, and Microsoft.
--------------------------------------------------

--- Chunk 2 ---
Source: docs\Google.txt
Length: 867 characters
Content:
Google was founded on September 4, 1998, by
American computer scientists Larry Page and Sergey
Brin. Together, they own about 14% of its publicly
listed shares and control 56% of its stockholder voting


In [6]:
# step 3 : Create and persist ChromaDB vector store
persist_directory="db/chroma_db"

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    
# Create ChromaDB vector store
print("--- Creating vector store ---")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_directory, 
    collection_metadata={"hnsw:space": "cosine"}
)
print("--- Finished creating vector store ---")
    
print(f"Vector store created and saved to {persist_directory}")

c:\Users\Mallem Kondaiah\OneDrive\Documents\RAG\RAG_IMPLEMENTATION\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Mallem Kondaiah\OneDrive\Documents\RAG\RAG_IMPLEMENTATION\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mallem Kondaiah\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you e

--- Creating vector store ---
--- Finished creating vector store ---
Vector store created and saved to db/chroma_db


2. Retrievel Pipeline

In [8]:
persistent_directory = "db/chroma_db"

# Load embeddings and vector store
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db = Chroma(
    persist_directory=persistent_directory,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}  
)

# Search for relevant documents
query = "What was NVIDIA's first graphics accelerator called?"

retriever = db.as_retriever(search_kwargs={"k": 5})

# retriever = db.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={
#         "k": 5,
#         "score_threshold": 0.3  # Only return chunks with cosine similarity ≥ 0.3
#     }
# )

relevant_docs = retriever.invoke(query)

print(f"User Query: {query}")
# Display results
print("--- Context ---")
for i, doc in enumerate(relevant_docs, 1):
    print(f"Document {i}:\n{doc.page_content}\n")


# Synthetic Questions: 

# 1. "What was NVIDIA's first graphics accelerator called?"
# 2. "Which company did NVIDIA acquire to enter the mobile processor market?"
# 3. "What was Microsoft's first hardware product release?"
# 4. "How much did Microsoft pay to acquire GitHub?"
# 5. "In what year did Tesla begin production of the Roadster?"
# 6. "Who succeeded Ze'ev Drori as CEO in October 2008?"
# 7. "What was the name of the autonomous spaceport drone ship that achieved the first successful sea landing?"
# 8. "What was the original name of Microsoft before it became Microsoft?"

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2031.71it/s]


User Query: What was NVIDIA's first graphics accelerator called?
--- Context ---
Document 1:
building in 2018
in the "next version" of the GX graphics chips which he had
worked on at Sun.[32] Then Huang told Priem to "drop the
GX", resulting in the name "NV".[32] Priem made a list of words with the letters "NV" in them.[34] At one
point, Malachowsky and Priem wanted to call the company NVision, but that name was already taken by
a manufacturer of toilet paper.[26] Both Priem[34] and Huang have taken credit for coming up with the
name Nvidia,[26] from "invidia", the Latin word for "envy".[29]

After the company outgrew Priem's townhouse, its original headquarters office was in Sunnyvale,
California.[29]

First graphics accelerator
Nvidia's first graphics accelerator, the NV1, was designed to process quadrilateral primitives (forward
texture mapping), a feature that set it apart from competitors, who preferred triangle primitives.[26]

Document 2:
﻿Nvidia
Nvidia Corporation[a] (/ɛnˈvɪdiə

In [14]:
persistent_directory = "db/chroma_db"

# 1. Load Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Load DB
db = Chroma(
    persist_directory=persistent_directory,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}  
)

# 3. Retrieve Documents (Reducing k to 3 to save tokens on free tier)
query = "What was the name of the autonomous spaceport drone ship that achieved the first successful sea landing?"
retriever = db.as_retriever(search_kwargs={"k": 3})
relevant_docs = retriever.invoke(query)

# 4. Initialize Gemini (Updated model name for 2026)
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0,
    max_retries=6,  # Automatically retries if you hit a rate limit
    convert_system_message_to_human=True # Better compatibility for Gemini
)

# 5. Prepare Input
combined_input = f"""Based on the following documents, answer this question: {query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in relevant_docs])}

If the answer isn't in the documents, say "I don't have enough information."
"""

messages = [
    SystemMessage(content="You are a helpful assistant specialized in Microsoft and GitHub history."),
    HumanMessage(content=combined_input),
]

# 6. Invoke and Print
try:
    result = model.invoke(messages)
    print("\n--- Generated Response (Gemini) ---")
    print(result.content)
except Exception as e:
    print(f"Error: {e}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2428.91it/s]



--- Generated Response (Gemini) ---
The autonomous spaceport drone ship (ASDS) that achieved the first successful sea landing was named **Of Course I Still Love You**.


In [5]:


# 1. Setup Models
# Make sure your database was built with this SAME embedding model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
persistent_directory = "db/chroma_db"

db = Chroma(persist_directory=persistent_directory, embedding_function=embedding_model)

# Use Gemini 1.5 Flash (or 2.0/2.5 if available in your region)
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0,
    max_retries=6,  # Automatically retries if you hit a rate limit
    convert_system_message_to_human=True # Better compatibility for Gemini
)

# 2. Conversation History
chat_history = []

def ask_question(user_question):
    global chat_history
    print(f"\n--- Processing: {user_question} ---")
    
    # STEP 1: Contextualize the question
    # This turns "How much did they pay?" into "How much did Microsoft pay for GitHub?"
    search_question = user_question
    if chat_history:
        context_prompt = [
            SystemMessage(content="Given the chat history and a new question, rewrite it as a standalone question that can be understood without the history. Just return the text of the new question."),
        ] + chat_history + [HumanMessage(content=f"Rewrite this question: {user_question}")]
        
        result = model.invoke(context_prompt)
        search_question = result.content.strip()
        print(f"Standalone Search Query: {search_question}")

    # STEP 2: Retrieve Documents
    retriever = db.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(search_question)
    
    # STEP 3: Generate Answer with Context
    context_text = "\n".join([f"- {doc.page_content}" for doc in docs])
    combined_input = f"""Answer the question using ONLY the provided documents.
    
    Documents:
    {context_text}
    
    Question: {user_question}
    """

    messages = [
        SystemMessage(content="You are a helpful assistant. Use the provided context to answer questions accurately."),
    ] + chat_history + [HumanMessage(content=combined_input)]
    
    response = model.invoke(messages)
    answer = response.content
    
    # STEP 4: Update History
    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=answer))
    
    # Keep history manageable (last 6 messages)
    if len(chat_history) > 10:
        chat_history = chat_history[-10:]
        
    print(f"\nAnswer: {answer}")

def start_chat():
    print("Welcome to your RAG Chat! (Type 'quit' to exit)")
    while True:
        user_input = input("\nYour question: ")
        if user_input.lower() == 'quit':
            break
        ask_question(user_input)

if __name__ == "__main__":
    start_chat()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2136.78it/s]


Welcome to your RAG Chat! (Type 'quit' to exit)

--- Processing: In what year did Tesla begin production of the Roadster? ---

Answer: Tesla began production of the Roadster in 2008.

--- Processing: how much cost it is ---
Standalone Search Query: How much does it cost?

Answer: Based on the provided documents:

*   The development cost for the Falcon 9 launch vehicle was approximately $300 million.
*   Approximately $90 million was spent developing the Falcon 1 launch vehicle.
*   The total development cost for Falcon 1 and Falcon 9 was $390 million.
*   The cost of each Platinum membership to the Linux Foundation is US$500,000 per year.
*   Intune for Education is priced at $30 per device.

--- Processing: what is the total cost ---


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}